# PAD-ONAP Kaggle Notebook: ML Baselines without XGBoost


This notebook is designed for Kaggle and CICDDoS2019-style CSV files. It is safe: it reads offline CSV data and does not generate packet traffic. Set PAD_ONAP_DATASET_DIR if auto-discovery under /kaggle/input is not enough.

Outputs are written under /kaggle/working so they can be downloaded as thesis evidence. Do not report any metric in the thesis unless it is produced by an executed notebook run and saved in the output artifacts.


This notebook trains classical ML baselines that do not use XGBoost: Random Forest, Extra Trees, HistGradientBoosting, logistic SGD, and MLP. It uses the 17 window-level features from the ddos-train-v4 AI direction.


In [ ]:
import os
import json
import time
import math
import glob
import pickle
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

DATASET_DIR = Path(os.environ.get("PAD_ONAP_DATASET_DIR", "/kaggle/input"))
if not DATASET_DIR.exists():
    DATASET_DIR = Path(".")

OUTPUT_DIR = Path("/kaggle/working/pad_onap_outputs")
MODELS_DIR = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
for d in [OUTPUT_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = int(os.environ.get("PAD_ONAP_CHUNKSIZE", "200000"))
WINDOW_SIZE = int(os.environ.get("PAD_ONAP_WINDOW_SIZE", "100"))
STEP = int(os.environ.get("PAD_ONAP_STEP", "50"))
BENIGN_STEP = int(os.environ.get("PAD_ONAP_BENIGN_STEP", "5"))
MAX_WINDOWS = int(os.environ.get("PAD_ONAP_MAX_WINDOWS", "120000"))
MIN_CLASS_WINDOWS = int(os.environ.get("PAD_ONAP_MIN_CLASS_WINDOWS", "10000"))

CLASS_NAMES = {
    0: "Normal",
    1: "UDP_Flood",
    2: "SYN_Flood",
    3: "HTTP_Flood",
    4: "ICMP_Flood",
    5: "Amplification",
    6: "Slow_rate",
}

FEATURE_NAMES = [
    "pkt_rate", "byte_rate", "src_ip_entropy", "dst_ip_entropy",
    "src_port_entropy", "dst_port_entropy", "proto_dist_tcp",
    "proto_dist_udp", "proto_dist_icmp", "syn_ratio", "fin_ratio",
    "avg_pkt_size", "pkt_size_std", "new_flows_rate",
    "flow_duration_mean", "inter_arrival_mean", "inter_arrival_std",
]

def normalize_col(name):
    return "".join(ch for ch in str(name).lower().strip() if ch.isalnum())

def col_lookup(df):
    return {normalize_col(c): c for c in df.columns}

def get_col(df, names, default=0.0):
    lookup = col_lookup(df)
    for name in names:
        key = normalize_col(name)
        if key in lookup:
            return pd.to_numeric(df[lookup[key]], errors="coerce").fillna(default).to_numpy()
    return np.full(len(df), default, dtype=np.float64)

def get_str_col(df, names):
    lookup = col_lookup(df)
    for name in names:
        key = normalize_col(name)
        if key in lookup:
            return df[lookup[key]].astype(str).fillna("").to_numpy()
    return np.array([""] * len(df), dtype=object)

def shannon_entropy(values):
    if len(values) == 0:
        return 0.0
    _, counts = np.unique(values, return_counts=True)
    probs = counts.astype(np.float64) / max(1, counts.sum())
    return float(-(probs * np.log2(probs + 1e-12)).sum())

def map_label(label):
    s = str(label).strip().lower()
    s_norm = s.replace("-", "_").replace(" ", "_")
    if s_norm in {"benign", "normal"}:
        return 0
    if "udp" in s_norm and "lag" not in s_norm:
        return 1
    if "udp_lag" in s_norm or "udplag" in s_norm:
        return 1
    if "syn" in s_norm:
        return 2
    if "webddos" in s_norm or "http" in s_norm:
        return 3
    if "icmp" in s_norm:
        return 4
    if any(x in s_norm for x in ["drdos", "dns", "ldap", "mssql", "netbios", "ntp", "snmp", "ssdp", "tftp", "portmap", "chargen", "amplification"]):
        return 5
    if any(x in s_norm for x in ["slowloris", "slowhttptest", "slow_rate", "slowrate"]):
        return 6
    return None

def find_label_column(df):
    lookup = col_lookup(df)
    for candidate in ["Label", "label", "Class", "class"]:
        key = normalize_col(candidate)
        if key in lookup:
            return lookup[key]
    return None

def extract_17_features(df_window):
    n = len(df_window)
    if n == 0:
        return np.zeros(len(FEATURE_NAMES), dtype=np.float32)

    dur = get_col(df_window, ["Flow Duration", "FlowDuration"], 1.0)
    dur = np.where(dur <= 0, 1.0, dur)
    dur_sum = max(float(np.sum(dur)), 1.0)

    fwd_pkts = get_col(df_window, ["Total Fwd Packets", "Tot Fwd Pkts", "TotalFwdPackets"], 0.0)
    bwd_pkts = get_col(df_window, ["Total Backward Packets", "Total Bwd packets", "Tot Bwd Pkts"], 0.0)
    total_pkts = fwd_pkts + bwd_pkts

    fwd_bytes = get_col(df_window, ["Total Length of Fwd Packets", "TotLen Fwd Pkts", "Total Length Fwd Packets"], 0.0)
    bwd_bytes = get_col(df_window, ["Total Length of Bwd Packets", "TotLen Bwd Pkts", "Total Length Bwd Packets"], 0.0)
    total_bytes = fwd_bytes + bwd_bytes

    pkt_rate = float(np.sum(total_pkts) / dur_sum * 1e6)
    byte_rate = float(np.sum(total_bytes) / dur_sum * 1e6)

    src_ip_entropy = shannon_entropy(get_str_col(df_window, ["Source IP", "Src IP", "SrcIP"]))
    dst_ip_entropy = shannon_entropy(get_str_col(df_window, ["Destination IP", "Dst IP", "DstIP"]))
    src_port_entropy = shannon_entropy(get_str_col(df_window, ["Source Port", "Src Port", "SrcPort"]))
    dst_port_entropy = shannon_entropy(get_str_col(df_window, ["Destination Port", "Dst Port", "DstPort"]))

    proto = get_col(df_window, ["Protocol"], 6).astype(int)
    proto_dist_tcp = float((proto == 6).sum() / max(n, 1))
    proto_dist_udp = float((proto == 17).sum() / max(n, 1))
    proto_dist_icmp = float((proto == 1).sum() / max(n, 1))

    syn_count = get_col(df_window, ["SYN Flag Count", "SYNFlagCount"], 0.0)
    fin_count = get_col(df_window, ["FIN Flag Count", "FINFlagCount"], 0.0)
    pkt_sum = max(float(np.sum(total_pkts)), 1.0)
    syn_ratio = float(np.sum(syn_count) / pkt_sum)
    fin_ratio = float(np.sum(fin_count) / pkt_sum)

    avg_pkt_size_raw = get_col(df_window, ["Average Packet Size", "Avg Packet Size"], np.nan)
    if np.isfinite(avg_pkt_size_raw).any():
        avg_pkt_size = float(np.nanmean(avg_pkt_size_raw))
    else:
        avg_pkt_size = float(np.sum(total_bytes) / pkt_sum)

    pkt_size_std_raw = get_col(df_window, ["Packet Length Std", "Pkt Len Std"], np.nan)
    if np.isfinite(pkt_size_std_raw).any():
        pkt_size_std = float(np.nanmean(pkt_size_std_raw))
    else:
        pkt_size_std = float(np.std(total_bytes / np.maximum(total_pkts, 1.0)))

    new_flows_rate = float(n / dur_sum * 1e6)
    flow_duration_mean = float(np.mean(dur))
    inter_arrival_mean = float(np.mean(get_col(df_window, ["Flow IAT Mean", "FlowIATMean"], 0.0)))
    inter_arrival_std = float(np.mean(get_col(df_window, ["Flow IAT Std", "FlowIATStd"], 0.0)))

    feats = np.array([
        pkt_rate, byte_rate, src_ip_entropy, dst_ip_entropy,
        src_port_entropy, dst_port_entropy, proto_dist_tcp,
        proto_dist_udp, proto_dist_icmp, syn_ratio, fin_ratio,
        avg_pkt_size, pkt_size_std, new_flows_rate,
        flow_duration_mean, inter_arrival_mean, inter_arrival_std,
    ], dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=1e9, neginf=-1e9)
    return feats

def audit_csv_files(csv_files):
    label_counts = defaultdict(int)
    total_rows = 0
    for f in csv_files:
        try:
            for chunk in pd.read_csv(f, chunksize=CHUNKSIZE, low_memory=False, on_bad_lines="skip"):
                lcol = find_label_column(chunk)
                if lcol is None:
                    continue
                total_rows += len(chunk)
                mapped = chunk[lcol].map(map_label)
                for cls, cnt in mapped.dropna().astype(int).value_counts().items():
                    label_counts[int(cls)] += int(cnt)
        except Exception as exc:
            print(f"Audit skipped {f}: {exc}")
    return total_rows, dict(label_counts)

def build_windows(csv_files, max_windows=MAX_WINDOWS):
    X, y, segments = [], [], []
    class_windows = defaultdict(int)
    active = set(range(7))
    class_quota = {c: max(MIN_CLASS_WINDOWS, max_windows // max(1, len(active))) for c in active}
    class_quota[0] = max(class_quota[0], MIN_CLASS_WINDOWS)
    t0 = time.time()

    for file_idx, f in enumerate(csv_files):
        if len(X) >= max_windows:
            break
        file_count = 0
        carry = pd.DataFrame()
        try:
            for chunk in pd.read_csv(f, chunksize=CHUNKSIZE, low_memory=False, on_bad_lines="skip"):
                lcol = find_label_column(chunk)
                if lcol is None:
                    continue
                mapped = chunk[lcol].map(map_label)
                chunk = chunk.loc[mapped.notna()].copy()
                if len(chunk) == 0:
                    continue
                chunk["_class"] = mapped.loc[mapped.notna()].astype(int).to_numpy()
                df = pd.concat([carry, chunk], ignore_index=True)

                i = 0
                while i + WINDOW_SIZE <= len(df) and len(X) < max_windows:
                    w = df.iloc[i:i + WINDOW_SIZE]
                    counts = np.bincount(w["_class"].to_numpy(dtype=np.int64), minlength=7)
                    cls = int(np.argmax(counts))
                    step = BENIGN_STEP if cls == 0 else STEP
                    if class_windows[cls] < class_quota.get(cls, max_windows):
                        X.append(extract_17_features(w))
                        y.append(cls)
                        class_windows[cls] += 1
                        file_count += 1
                    i += step
                carry = df.iloc[i:].reset_index(drop=True)
        except Exception as exc:
            print(f"Window extraction skipped {f}: {exc}")
        if file_count > 0:
            segments.append(file_count)
        print(f"[{file_idx + 1}/{len(csv_files)}] {Path(f).name}: windows={file_count}, total={len(X)}, elapsed={time.time() - t0:.1f}s")

    if len(X) == 0:
        raise RuntimeError("No windows extracted. Check dataset path and label mapping.")

    return np.vstack(X).astype(np.float32), np.asarray(y, dtype=np.int64), segments

csv_files = sorted(glob.glob(str(DATASET_DIR / "**" / "*.csv"), recursive=True))
print(f"DATASET_DIR={DATASET_DIR}")
print(f"CSV files found: {len(csv_files)}")
for p in csv_files[:10]:
    print(" -", p)
if len(csv_files) > 10:
    print(f" ... {len(csv_files) - 10} more")


## Build feature windows

This cell scans CICDDoS2019-style CSV files and extracts 17 window-level features. You can reduce runtime by setting PAD_ONAP_MAX_WINDOWS in Kaggle notebook environment variables.


In [ ]:
total_rows, label_counts = audit_csv_files(csv_files)
print("Total audited rows:", total_rows)
print("Mapped label counts:")
for cls in range(7):
    print(f"  {cls} {CLASS_NAMES[cls]:<15}: {label_counts.get(cls, 0):,}")

X, y, segments = build_windows(csv_files, MAX_WINDOWS)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("segment count:", len(segments))

np.save(OUTPUT_DIR / "features_X.npy", X)
np.save(OUTPUT_DIR / "features_y.npy", y)
with open(OUTPUT_DIR / "feature_metadata.json", "w") as f:
    json.dump({
        "feature_names": FEATURE_NAMES,
        "class_names": CLASS_NAMES,
        "window_size": WINDOW_SIZE,
        "step": STEP,
        "benign_step": BENIGN_STEP,
        "max_windows": MAX_WINDOWS,
        "segments": segments,
        "note": "17 window-level features; safe offline dataset processing; no packet traffic generated."
    }, f, indent=2)


## Train non-XGBoost classifiers

The models are intentionally limited to scikit-learn methods so the notebook can run on Kaggle without installing extra packages.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

models = {
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_SEED
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=31,
        l2_regularization=0.01, random_state=RANDOM_SEED
    ),
    "sgd_logistic": make_pipeline(
        StandardScaler(),
        SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=2000,
                      class_weight="balanced", random_state=RANDOM_SEED)
    ),
    "mlp": make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",
                      alpha=1e-4, learning_rate_init=1e-3, batch_size=512,
                      max_iter=80, early_stopping=True, random_state=RANDOM_SEED)
    ),
}

results = []
reports = {}

for name, model in models.items():
    print("\n" + "=" * 72)
    print("Training", name)
    t0 = time.time()
    if name == "hist_gradient_boosting":
        model.fit(X_train, y_train, sample_weight=sample_weight)
    else:
        model.fit(X_train, y_train)
    train_time = time.time() - t0

    t1 = time.time()
    y_pred = model.predict(X_test)
    infer_time = time.time() - t1

    metrics = {
        "model": name,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
        "train_time_sec": float(train_time),
        "test_inference_time_sec": float(infer_time),
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
    }
    results.append(metrics)
    print(json.dumps(metrics, indent=2))

    labels_present = sorted(np.unique(np.concatenate([y_test, y_pred])))
    target_names = [CLASS_NAMES[i] for i in labels_present]
    report = classification_report(
        y_test, y_pred, labels=labels_present, target_names=target_names,
        digits=4, zero_division=0
    )
    reports[name] = report
    print(report)

    cm = confusion_matrix(y_test, y_pred, labels=labels_present)
    np.savetxt(TABLES_DIR / f"{name}_confusion_matrix.csv", cm, delimiter=",", fmt="%d")
    joblib.dump(model, MODELS_DIR / f"{name}.joblib")

df_results = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
df_results.to_csv(TABLES_DIR / "ml_baseline_results.csv", index=False)
with open(OUTPUT_DIR / "ml_baseline_reports.txt", "w") as f:
    for name, report in reports.items():
        f.write("\n" + "=" * 72 + "\n")
        f.write(name + "\n")
        f.write(report + "\n")
with open(OUTPUT_DIR / "ml_baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

df_results


## Plot and export thesis-ready summary

Only use these values in the thesis after the notebook has been executed and the output files have been saved.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
plot_df = df_results.sort_values("macro_f1")
ax.barh(plot_df["model"], plot_df["macro_f1"], color="#4C78A8")
ax.set_xlabel("Macro F1")
ax.set_title("Non-XGBoost ML baselines on PAD-ONAP window features")
ax.set_xlim(0, 1.0)
for i, v in enumerate(plot_df["macro_f1"]):
    ax.text(v + 0.01, i, f"{v:.4f}", va="center")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ml_baseline_macro_f1.png", dpi=160)
plt.show()

best_name = df_results.iloc[0]["model"]
summary = [
    "# PAD-ONAP non-XGBoost ML baseline summary",
    "",
    f"Best model by macro F1: {best_name}",
    "",
    "This notebook uses 17 window-level entropy, rate, protocol, flag, packet-size, and timing features.",
    "It does not use XGBoost and does not generate packet traffic.",
    "",
    "## Metrics",
    df_results.to_markdown(index=False),
]
with open(OUTPUT_DIR / "ml_baseline_summary.md", "w") as f:
    f.write("\n".join(summary))
print("\n".join(summary[:8]))
print("Outputs:", OUTPUT_DIR)
